In [ ]:
from TerraFin import configure, load_terrafin_config
from TerraFin.data import DataFactory
from TerraFin.data.utils.filters import date_filter


configure()
config = load_terrafin_config()

test_data = DataFactory().get_market_data("NVDA")
test_data = date_filter(test_data, "2024-01-01", "2025-12-31")
closes = test_data.close.astype(float).tolist()

### 1. Power Spectrum (Periodogram)
Identify which periodic cycles exist in the price data and how strong they are.

In [ ]:
import plotly.graph_objects as go

from TerraFin.analytics.analysis.technical import power_spectrum


periods, power = power_spectrum(closes)

fig = go.Figure()
fig.add_trace(go.Scatter(x=periods, y=power, mode="lines", name="Power"))
fig.update_layout(
    title="Power Spectrum of Log Returns",
    xaxis_title="Period (trading days)",
    yaxis_title="Power",
    xaxis={"range": [2, len(closes) // 2]},
)
fig.show()

### 2. Dominant Cycles
Extract the top periodic cycles ranked by signal-to-noise ratio.

In [ ]:
import pandas as pd

from TerraFin.analytics.analysis.technical import dominant_cycles


cycles = dominant_cycles(closes, top_n=5)

df_cycles = pd.DataFrame(cycles, columns=["Period (days)", "Power", "SNR"])
df_cycles.index = range(1, len(df_cycles) + 1)
df_cycles.index.name = "Rank"
df_cycles

### 3. Amplitude & Phase
Amplitude = how large each cycle's swing is.  
Phase = where in the cycle we currently are.

In [ ]:
from TerraFin.analytics.analysis.technical import amplitude_phase


periods, amplitudes, phases = amplitude_phase(closes)

fig = go.Figure()
fig.add_trace(go.Bar(x=periods[:50], y=amplitudes[:50], name="Amplitude"))
fig.update_layout(
    title="Amplitude Spectrum (top 50 lowest frequencies)",
    xaxis_title="Period (trading days)",
    yaxis_title="Amplitude (return magnitude)",
)
fig.show()

### 4. Spectral Filtering
Keep only cycles between 10 and 60 days, removing short-term noise and long-term trend.

In [ ]:
import numpy as np

from TerraFin.analytics.analysis.technical import spectral_filter


log_returns = list(np.diff(np.log(closes)))
filtered = spectral_filter(closes, min_period=10, max_period=60)

fig = go.Figure()
fig.add_trace(go.Scatter(y=log_returns, mode="lines", name="Original", opacity=0.35))
fig.add_trace(go.Scatter(y=filtered, mode="lines", name="Filtered (10-60 day cycles)", line=dict(width=2)))
fig.update_layout(
    title="Spectral Band-Pass Filter: Original vs Filtered Returns",
    xaxis_title="Trading Day",
    yaxis_title="Log Return",
)
fig.show()

### 5. Spectrogram (Time-Frequency Heatmap)
See how the dominant cycles evolve over time — reveals regime changes.

In [ ]:
from TerraFin.analytics.analysis.technical import spectrogram


periods_sg, time_indices, power_matrix = spectrogram(closes, segment_size=64, overlap=48)

# Clip periods to a readable range (2-64 days)
max_display_period = 64
display_cols = [i for i, p in enumerate(periods_sg) if p <= max_display_period]
display_periods = [periods_sg[i] for i in display_cols]
display_matrix = [[row[i] for i in display_cols] for row in power_matrix]

fig = go.Figure(
    data=go.Heatmap(
        z=display_matrix,
        x=[f"{p:.0f}d" for p in display_periods],
        y=time_indices,
        colorscale="Hot",
        colorbar={"title": "Power"},
    )
)
fig.update_layout(
    title="Spectrogram: Cycle Strength Over Time",
    xaxis_title="Period (trading days)",
    yaxis_title="Center Candle Index",
)
fig.show()